### Imbalance Handling

Goal: To test different imbalance handling techniques on RF model (it performed better than standard Logistic Regression), to improve model PR-AUC - a metric suited for imbalanced datasets where precision and recall are both considered

Techniques tested:

- class_weight='balanced' - increases the importance of the minority class during training
- SMOTE - generates more churn samples to balance the training data
- cost-sensitive learning - Optuna looking for optimal weights
- XGBoost + LightGBM with scale_pos_weight
- Optuna tuning on the best model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.metrics import confusion_matrix
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import optuna

In [ ]:
df = pd.read_csv("../../data/Customer-Churn-Records.csv")

In [ ]:
X = df.drop(columns=["Complain", "Exited", "RowNumber", "CustomerId", "Surname"])
y = df["Exited"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

### RF Baseline (reference)
PR-AUC ~0.67 from `01_baseline.ipynb`. Used as reference point for all experiments below.

In [ ]:
categorical = X.select_dtypes(include="object").columns
numerical = X.select_dtypes(exclude="object").columns

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
        ('scaler', StandardScaler(), numerical)
    ],
    remainder='passthrough'
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
rf_balanced = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1))
])

scores = cross_validate(rf_balanced, X_train, y_train, cv=skf, scoring=['average_precision'])
print(f"RF balanced PR-AUC: {scores['test_average_precision'].mean():.3f}")

In [ ]:
def objective(trial):
    weight = trial.suggest_float('weight', 1.0, 20.0)
    pipeline = Pipeline([
        ('prep', preprocessor),
        ('model', RandomForestClassifier(class_weight={0: 1, 1: weight}, random_state=42, n_jobs=-1))
    ])

    scores = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=['average_precision'])
    return scores['test_average_precision'].mean()

study = optuna.create_study(direction="maximize")
optuna.logging.set_verbosity(optuna.logging.WARNING)
study.optimize(objective, n_trials=30)

print(f"Best weight: {study.best_params['weight']:.2f}")
print(f"Best PR-AUC: {study.best_value:.3f}")

In [ ]:
rf_smote = Pipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, n_jobs=-1))
])

scores = cross_validate(rf_smote, X_train, y_train, cv=skf, scoring=['average_precision'])
print(f"RF SMOTE PR-AUC: {scores['test_average_precision'].mean():.3f}")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

pipeline_XGB = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(scale_pos_weight=scale_pos_weight, random_state=42))
])

pipeline_LGBM = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1))
])

scores = cross_validate(pipeline_XGB, X_train, y_train, cv=skf, scoring=['average_precision'])
print(f"XGB balanced PR-AUC: {scores['test_average_precision'].mean():.3f}")

scores = cross_validate(pipeline_LGBM, X_train, y_train, cv=skf, scoring=['average_precision'])
print(f"LGBM balanced PR-AUC: {scores['test_average_precision'].mean():.3f}")

In [ ]:
def objective(trial):
    params = {
            'n_estimators': trial.suggest_int('n_estimators', 100, 800),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        }
    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(**params, scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1))
    ])

    scores = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=['average_precision'])
    return scores['test_average_precision'].mean()

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(f"Best weight: {study.best_params}")
print(f"Best PR-AUC: {study.best_value:.3f}")

### Results

| Model | PR-AUC |
|---|---|
| RF baseline | 0.670 |
| RF balanced | 0.655 |
| RF cost-sensitive | 0.672 |
| RF SMOTE |  0.642 |
| XGB balanced | 0.655 |
| LGBM balanced | 0.680 |
| **LGBM tuned (Optuna)** | **0.702** |

Imbalance handling techniques did not improve Random Forest performance. The baseline model achieved a PR-AUC of 0.670. A cost-sensitive Random Forest, where the class weight was optimized, slightly improved the result to 0.672. In contrast, Balanced Random Forest (0.655) and SMOTE (0.642) performed worse than the baseline model.

The best model was LightGBM. Even without tuning, it achieved a PR-AUC of 0.680, outperforming both Random Forest and XGBoost.

After hyperparameter tuning with Optuna, LightGBM reached a PR-AUC of 0.702, which was the highest score among all tested models.

In [ ]:
# Best params from Optuna (50 trials), PR-AUC: 0.702
best_params = {
    'n_estimators': 630,
    'max_depth': 4,
    'learning_rate': 0.0168,
    'subsample': 0.735,
    'colsample_bytree': 0.636,
    'reg_alpha': 2.25e-05,
    'reg_lambda': 8.07e-08
}

pipeline_optuna_lgbm = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', LGBMClassifier(**best_params, scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1))
    ])

pipeline_optuna_lgbm.fit(X_train, y_train)

In [ ]:
y_prob = pipeline_optuna_lgbm.predict_proba(X_test)[:, 1]

from sklearn.calibration import calibration_curve

fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_prob, n_bins=20)

plt.plot(mean_predicted_value, fraction_of_positives, label='LGBM')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect calibration')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration Curve')
plt.legend()
plt.savefig('../../reports/figures/model_calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()

### Model Calibration Analysis

The calibration curve indicates that the LightGBM model is generally overconfident, as most points lie below the diagonal reference line. This means that the predicted probabilities are often higher than the observed churn rates, particularly in the mid-probability range.

While this does not significantly affect the model's ability to rank customers by churn risk, it may lead to inaccurate interpretation of the predicted probabilities. Therefore, the model remains useful for identifying high-risk customers, but probability calibration could improve the reliability of churn estimates for business decision-making.